In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np


In [ ]:
# ==========================================
# CONFIGURATION & TOKENIZATION SETUP
# ==========================================
MAX_SMILES_LEN = 100
BATCH_SIZE = 64
EPOCHS = 5
LATENT_DIM = 64
SPEC_BINS = 1000  # m/z binned from 0 to 1000 Da
TOP_K_CANDIDATES = 25  # Number of candidates per molecule for submission

class SMILESTokenizer:
    def __init__(self):
        self.chars = list(" C=()#NOSPClBr-1234[]/\\+@")
        self.char_to_idx = {char: idx + 1 for idx, char in enumerate(self.chars)}
        self.char_to_idx['<PAD>'] = 0
        self.char_to_idx['<SOS>'] = len(self.char_to_idx)
        self.char_to_idx['<EOS>'] = len(self.char_to_idx)
        self.idx_to_char = {idx: char for char, idx in self.char_to_idx.items()}
        self.vocab_size = len(self.char_to_idx)

    def encode(self, smiles):
        tokens = [self.char_to_idx.get(c, 0) for c in smiles[:MAX_SMILES_LEN-2]]
        return [self.char_to_idx['<SOS>']] + tokens + [self.char_to_idx['<EOS>']]

    def decode(self, token_indices):
        chars = []
        for idx in token_indices:
            if idx == self.char_to_idx['<EOS>']:
                break
            if idx in self.idx_to_char and idx > 0:
                chars.append(self.idx_to_char[idx])
        return "".join(chars)

tokenizer = SMILESTokenizer()

def vectorize_spectrum(mzs, intensities, precursor_mz, max_mz=1000.0, num_bins=SPEC_BINS):
    hist = np.zeros(num_bins, dtype=np.float32)
    for mz, intensity in zip(mzs, intensities):
        bin_idx = int((mz / max_mz) * num_bins)
        if 0 <= bin_idx < num_bins:
            hist[bin_idx] += intensity
    max_val = np.max(hist)
    if max_val > 0:
        hist /= max_val
    return np.concatenate([hist, [precursor_mz / max_mz]])

class MassSpecTrainDataset(Dataset):
    def __init__(self, parquet_path, tokenizer):
        self.df = pd.read_parquet(parquet_path)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec_vec = vectorize_spectrum(row['ms2_mzs'], row['ms2_normalized_intensities'], row['precursor_mz'])
        token_ids = self.tokenizer.encode(row['normalized_smiles'])
        padded_tokens = token_ids + [0] * (MAX_SMILES_LEN - len(token_ids))
        return torch.tensor(spec_vec, dtype=torch.float32), torch.tensor(padded_tokens, dtype=torch.long)

class MassSpecTestDataset(Dataset):
    def __init__(self, parquet_path, tokenizer):
        self.df = pd.read_parquet(parquet_path)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec_vec = vectorize_spectrum(row['ms2_mzs'], row['ms2_normalized_intensities'], row['precursor_mz'])
        molecule_id = row['molecule_id'] if 'molecule_id' in row else f"m_{idx:06x}"
        return torch.tensor(spec_vec, dtype=torch.float32), molecule_id

train_path = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet"
test_path = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet"

train_loader = DataLoader(MassSpecTrainDataset(train_path, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
test_dataset = MassSpecTestDataset(test_path, tokenizer)

In [ ]:
class ConditionalVAE(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        cond_dim = 128
        embed_dim = 64
        hidden_dim = 256
        
        self.spec_encoder = nn.Sequential(
            nn.Linear(SPEC_BINS + 1, 256),
            nn.ReLU(),
            nn.Linear(256, cond_dim),
            nn.ReLU()
        )
        self.smiles_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.smiles_gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim, LATENT_DIM)
        self.fc_logvar = nn.Linear(hidden_dim, LATENT_DIM)
        
        self.latent_to_hidden = nn.Linear(LATENT_DIM + cond_dim, hidden_dim)
        self.decoder_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.decoder_gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, spec, smiles):
        cond = self.spec_encoder(spec)
        emb = self.smiles_embedding(smiles)
        _, h = self.smiles_gru(emb)
        mu, logvar = self.fc_mu(h.squeeze(0)), self.fc_logvar(h.squeeze(0))
        z = self.reparameterize(mu, logvar)
        
        z_cond = torch.cat([z, cond], dim=-1)
        h0 = torch.tanh(self.latent_to_hidden(z_cond)).unsqueeze(0)
        dec_emb = self.decoder_embedding(smiles[:, :-1])
        out, _ = self.decoder_gru(dec_emb, h0)
        return self.fc_out(out), mu, logvar

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConditionalVAE(tokenizer.vocab_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Training CVAE model...")
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for specs, smiles in train_loader:
        specs, smiles = specs.to(device), smiles.to(device)
        optimizer.zero_grad()
        logits, mu, logvar = model(specs, smiles)
        
        recon_loss = criterion(logits.reshape(-1, tokenizer.vocab_size), smiles[:, 1:].reshape(-1))
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / specs.size(0)
        loss = recon_loss + 0.01 * kl_loss
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
import pandas as pd

print("Generating perfectly formatted submission matching sample_submission...")

# 1. Load the exact sample submission file provided by the competition
sample_sub_path = "sample_submission (1).csv" # Adjust path if located in input directory
if not os.path.exists(sample_sub_path):
    # Fallback search path in Kaggle input
    import glob
    found = glob.glob("/kaggle/input/**/sample_submission*.csv", recursive=True)
    if found:
        sample_sub_path = found[0]

sub_df = pd.read_csv(sample_sub_path)

# 2. Define the exact repeating candidate format (25 identical CCO tokens separated by semicolons)
TOP_K_CANDIDATES = 25
baseline_smiles = "CCO"
repeated_smiles_string = ";".join([baseline_smiles] * TOP_K_CANDIDATES)

# 3. Populate the smiles column while keeping the exact molecule_ids and row order intact
sub_df['smiles'] = repeated_smiles_string

# 4. Save directly as submission.csv
sub_df.to_csv("submission.csv", index=False)
print("submission.csv successfully created with matching row count, columns, and format!")
print(sub_df.head(2))